In [2]:
import pandas as pd
import seaborn as sns # new library 
import matplotlib.pyplot as plt
import numpy as np
from sklearn import feature_extraction, linear_model, model_selection, preprocessing
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [6]:
NLP_Train = pd.read_csv('data/NLP_train.csv')
NLP_Train.columns

Index(['id', 'keyword', 'location', 'text', 'target'], dtype='object')

In [7]:
NLP_Test = pd.read_csv('data/NLP_test.csv')
NLP_Test.shape
NLP_Test.columns

Index(['id', 'keyword', 'location', 'text'], dtype='object')

In [8]:
NLP_Train.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [9]:
NLP_Train[NLP_Train["target"]==1]["text"]

0       Our Deeds are the Reason of this #earthquake M...
1                  Forest fire near La Ronge Sask. Canada
2       All residents asked to 'shelter in place' are ...
3       13,000 people receive #wildfires evacuation or...
4       Just got sent this photo from Ruby #Alaska as ...
                              ...                        
7608    Two giant cranes holding a bridge collapse int...
7609    @aria_ahrary @TheTawniest The out of control w...
7610    M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...
7611    Police investigating after an e-bike collided ...
7612    The Latest: More Homes Razed by Northern Calif...
Name: text, Length: 3271, dtype: object

In [10]:
# Whats the number of negative to positive tweets?
Not_disaster= (NLP_Train[['target']]== 0).sum()
Disaster= (NLP_Train[['target']]== 1).sum()

print(f"Ratio of non-disaster tweets to disaster: {Not_disaster}: {Disaster}")
# we have a pretty symetrical split between tweet types

Ratio of non-disaster tweets to disaster: target    4342
dtype: int64: target    3271
dtype: int64


In [19]:
# count words in a tweet- a vector here is a set of numbers that ML can work with
# it will match common words (or tokens) for negative scenarios and positive ones attributing these as disaster tweets or non disaster ones

In [23]:
count_vectorizer = feature_extraction.text.CountVectorizer()

## let's get counts for the first 5 tweets in the data
example_train_vectors = count_vectorizer.fit_transform(NLP_Train["text"][0:5])

In [24]:
print(example_train_vectors[0].todense().shape)
print(example_train_vectors[0].todense())
# tells us there arr 54 unique tokens in first 5 tweets 
# The first sweet only contains 13 of these unique tokens

(1, 54)
[[0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0
  0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 0]]


In [26]:
train_vectors = count_vectorizer.fit_transform(NLP_Train["text"])

In [27]:
test_vectors = count_vectorizer.transform(NLP_Test["text"])

In [28]:
y_train = NLP_Train['target']
y_test = NLP_Test['target']

KeyError: 'target'

In [29]:
clf = linear_model.RidgeClassifier()

In [30]:
# Cross validation on a portion of our training data, vlaidate with the other half 
scores = model_selection.cross_val_score(clf, train_vectors, NLP_Train["target"], cv=3, scoring="f1")
print(f"Cross-validation accuracy: {scores.mean():.3f}± {scores.std():.4f}")

Cross-validation accuracy: 0.600± 0.0315


In [31]:
#Lets hyperparameter train to allow the best prediction model. 

y_train=NLP_Train["target"]

param_dist = {
    'alpha': np.logspace(-4, 4, 50),
    'fit_intercept': [False],
    'solver': ['auto', 'lsqr', 'sparse_cg', 'sag']
}

random_search = RandomizedSearchCV(
    estimator=clf,
    param_distributions=param_dist,
    n_iter=10,           # Number of random combos to try
    cv=5,                # 5-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1            # Use all cores
)

random_search.fit(train_vectors, y_train)
best_model = random_search.best_estimator_
test_predictions = best_model.predict(test_vectors)

print("Best Parameters:", random_search.best_params_)
print("Best Model Accuracy:", round(random_search.best_score_, 3))

#clf.fit(train_vectors, train_df["target"])

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'solver': 'sparse_cg', 'fit_intercept': False, 'alpha': np.float64(232.99518105153672)}
Best Model Accuracy: 0.709


In [32]:
Best_scores = model_selection.cross_val_score(best_model, train_vectors, NLP_Train["target"], cv=3, scoring="f1")
print(f"Cross-validation accuracy: {Best_scores.mean():.3f}± {scores.std():.4f}")
# our accuracy had only increase slightly.

Cross-validation accuracy: 0.641± 0.0315


In [33]:
best_model.fit(train_vectors, NLP_Train["target"])

RidgeClassifier(alpha=np.float64(232.99518105153672), fit_intercept=False,
                solver='sparse_cg')

In [34]:
sample_submission = pd.read_csv("data/sample_submission.csv")
sample_submission.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [37]:
sample_submission["target"] = best_model.predict(test_vectors)
# replacing the target column here with the models predicitons to generate a final submission fie

In [36]:
sample_submission.head()

,id,target
0,0,1
1,2,0
2,3,1
3,9,1
4,11,1


In [38]:
sample_submission.to_csv("NLP_Submission.csv", index= False)